# Bayesian A/B Testing: Posterior Evolution & Prior Sensitivity Analysis

This notebook demonstrates how Bayesian beliefs evolve as new evidence arrives using the Beta-Binomial conjugate model.

We visualize:

- Sequential Bayesian Updating
- Posterior Evolution
- Posterior Mean Trajectory
- Prior Sensitivity Analysis

The goal is to build intuition for Bayesian inference and understand how prior beliefs interact with observed data.

In [31]:
import numpy as np
from scipy.stats import beta as be
import plotly.graph_objects as go
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.BaysianABtest import BayesianABTest

## Problem Statement

Suppose we are running an A/B test on two webpage variants.

Instead of estimating a single conversion rate, Bayesian inference models uncertainty using probability distributions.

As more visitors arrive, our beliefs are updated through Bayes' theorem.

This notebook visualizes how the posterior distribution evolves over time and how different prior assumptions influence the analysis.

# Beta-Binomial Conjugacy

Suppose the prior distribution for the unknown probability of success \(\theta\) is

$$
\theta \sim \mathrm{Beta}(\alpha,\beta),
$$

with density

$$
P(\theta)
=
\frac{\Gamma(\alpha+\beta)}
{\Gamma(\alpha)\Gamma(\beta)}
\theta^{\alpha-1}(1-\theta)^{\beta-1},
\qquad 0<\theta<1.
$$

Assume that the observed data consist of \(x\) successes out of \(n\) independent Bernoulli trials. The likelihood is

$$
X \mid \theta \sim \mathrm{Binomial}(n,\theta),
$$

with probability mass function

$$
P(D\mid\theta)
=
\binom{n}{x}
\theta^{x}
(1-\theta)^{n-x}.
$$

Using Bayes' theorem,

$$
P(\theta\mid D)
=
\frac{P(D\mid\theta)P(\theta)}
{P(D)}
\propto
P(D\mid\theta)P(\theta).
$$

Substituting the likelihood and prior,

$$
\begin{aligned}
P(\theta\mid D)
&\propto
\theta^{x}(1-\theta)^{n-x}
\times
\theta^{\alpha-1}(1-\theta)^{\beta-1} \\
&=
\theta^{x+\alpha-1}
(1-\theta)^{n-x+\beta-1}.
\end{aligned}
$$

The resulting posterior has the same functional form as a Beta distribution. Hence,

$$
\boxed{
\theta \mid D
\sim
\mathrm{Beta}
\left(
\alpha+x,\,
\beta+n-x
\right)
}
$$

Thus, the Beta distribution is **conjugate** to the Binomial likelihood. The posterior parameters are simply updated by adding the observed successes and failures:

$$
\boxed{
\alpha_{\text{post}}=\alpha+x,
\qquad
\beta_{\text{post}}=\beta+n-x.
}
$$

## Sequential Bayesian Updating

Instead of waiting until an experiment finishes, Bayesian inference allows us to continuously update our beliefs as new observations arrive.

The posterior from one update becomes the prior for the next update.

This makes Bayesian A/B testing naturally suited for online experimentation.

In [32]:
theta = np.linspace(0, 1, 500)
scenarios = [
    (0, 0),
    (10, 2),
    (50, 11),
    (200, 42),
    (1000, 215),
]
## Priors
alpha_prior = 1
beta_prior = 1


In [34]:
frames = []
for visitors, conversions in scenarios:
    alpha_post = alpha_prior + conversions
    beta_post = beta_prior + visitors - conversions
    posterior_pdf = be.pdf(
    theta,
    alpha_post,
    beta_post,
    )
    frame = go.Frame(
    data=[
        go.Scatter(
            x=theta,
            y=posterior_pdf,
            mode="lines",
        )
    ],
    name=f"{visitors} Visitors",
)
    frames.append(frame)
    

In [ ]:
print(len(frames))

5


In [ ]:
initial_pdf = beta.pdf(theta, 1, 1)
fig = go.Figure(
    data=[
        go.Scatter(
            x=theta,
            y=initial_pdf,
            mode="lines",
            name="Posterior",
        )
    ],
    frames=frames,
)
fig.update_layout(
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {"duration": 800},
                            "fromcurrent": True,
                        },
                    ],
                }
            ],
        }
    ]
)
sliders = [
    {
        "steps": [
            {
                "method": "animate",
                "label": frame.name,
                "args": [
                    [frame.name],
                    {
                        "mode": "immediate",
                        "frame": {"duration": 500},
                    },
                ],
            }
            for frame in frames
        ]
    }
]

fig.update_layout(sliders=sliders,title="Sequential Bayesian Updating",
    xaxis_title="Conversion Rate",
    yaxis_title="Density")
fig.show()

# Goal

## This visualization answers:

How does our estimate of the conversion rate change as more data arrives?

Instead of plotting the full posterior every time, we'll plot:

Posterior Mean
95% Credible Interval
True Conversion Rate (reference line)

In [ ]:
daily_data = [
    (10, 2),
    (40, 9),
    (150, 31),
    (800, 173),
]

In [ ]:
test = BayesianABTest()
sample_sizes = []
means = []
lower_bounds = []
upper_bounds = []

In [ ]:
total_visitors = 0

for visitors, conversions in daily_data:

    total_visitors += visitors

    test.update(
        visitors_A=visitors,
        conversions_A=conversions,
        visitors_B=0,
        conversions_B=0,
    )

    summary = test.get_posterior_summary()

    sample_sizes.append(total_visitors)

    means.append(summary["A"]["mean"])

    lower, upper = summary["A"]["credible_interval"]

    lower_bounds.append(lower)

    upper_bounds.append(upper)

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=sample_sizes,
        y=means,
        mode="lines+markers",
        name="Posterior Mean",
    )
)

In [ ]:
fig.add_trace(
    go.Scatter(
        x=sample_sizes,
        y=upper_bounds,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=sample_sizes,
        y=lower_bounds,
        mode="lines",
        fill="tonexty",
        name="95% Credible Interval",
    )
)

In [ ]:
true_rate = 0.21 ## suppose
fig.add_hline(
    y=true_rate,
    line_dash="dash",
    annotation_text="True Conversion Rate",
)
fig.update_layout(
    title="Posterior Mean Over Time",
    xaxis_title="Total Visitors",
    yaxis_title="Conversion Rate",
)
fig.show()

# Goal

Show that different priors + same data produce different posteriors initially, but as more data is observed, the posteriors become increasingly similar.

This directly demonstrates:

Prior sensitivity
Posterior convergence
Data dominating the prior

In [ ]:
priors = {
    "Beta(1,1)": (1, 1),
    "Beta(2,2)": (2, 2),
    "Beta(9,91)": (9, 91),
}
visitors = 1000
conversions = 210

In [ ]:
theta = np.linspace(0, 1, 500)

In [35]:
fig = go.Figure()
for name, prior in priors.items():

    test = BayesianABTest(
        prior_params_A=prior,
    )

    test.update(
        visitors_A=visitors,
        conversions_A=conversions,
        visitors_B=0,
        conversions_B=0,
    )

    alpha, beta = test.posterior_params_A

    posterior = be.pdf(theta, alpha, beta)

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=posterior,
            mode="lines",
            name=name,
        )
    )
fig.update_layout(
    title="Prior Sensitivity Analysis",
    xaxis_title="Conversion Rate",
    yaxis_title="Posterior Density",
)
fig.show()

## Statistical Interpretation

Different priors influence the posterior when limited data is available. As the sample size increases, the likelihood dominates the prior, causing the posterior distributions to converge.

## Business Interpretation

Historical beliefs may influence early A/B test results. However, with sufficient user traffic, the observed data becomes the primary driver of inference, making conclusions robust to reasonable prior choices.

## Key Takeaway

Bayesian inference incorporates prior knowledge, but its influence naturally diminishes as more evidence is collected.

## Conclusion

In this notebook, we visualized how Bayesian inference updates beliefs through the Beta-Binomial model.

The posterior distribution evolved sequentially as additional observations were incorporated, demonstrating a reduction in uncertainty over time.

We also showed that while different priors can influence early inferences, their impact decreases with increasing sample size, leading to posterior convergence.

These visualizations provide intuition for why Bayesian A/B testing is well suited to continuous online experimentation.